# Week 1: data collection and coverage test

A thin runner: all logic lives in `pipeline/`, so this notebook and `make week1` do the same thing.
Runs locally (`make notebook`) or in Colab (first cell clones the repo).

Sections skip work already saved today, so re-running after an interruption is safe.

**Before committing:** outputs are stripped automatically by the pre-commit hook (`nbstripout`). Never paste posting text into markdown cells.

## Setup

In [ ]:
import os
import sys
import pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Colab: clone the repo, install deps, keep data on Drive
    from google.colab import drive, userdata, auth
    drive.mount('/content/drive')
    os.environ['AJT_DATA'] = '/content/drive/MyDrive/ai-job-titles-data'
    if not pathlib.Path('/content/ai-job-titles').exists():
        token = userdata.get('GITHUB_TOKEN')   # private repo: fine-grained token with read access
        !git clone -q https://{token}@github.com/ella-garcia/ai-job-titles.git /content/ai-job-titles
    os.chdir('/content/ai-job-titles')
    !git pull -q
    !pip install -q -r requirements.txt
    auth.authenticate_user()   # for BigQuery
else:
    # Local: run from the repo root (this notebook lives in notebooks/)
    if pathlib.Path.cwd().name == 'notebooks':
        os.chdir('..')
sys.path.insert(0, os.getcwd())

import pandas as pd
from pipeline.config import data_dir, run_dir, seed_everything
from pipeline.__main__ import record_environment
seed_everything(); record_environment()
pd.set_option('display.max_columns', 30)
print('Data folder:', data_dir())

## 1. Full job boards (Greenhouse, Lever, Ashby)

In [ ]:
from pipeline.boards import fetch_boards, board_titles
fetch_boards()
titles = board_titles()
pd.crosstab([titles.group, titles.company], titles.family)

## 2. Big employers by keyword search

Magnificent 7, consulting, large tech, telcos. About an hour, mostly Microsoft (rate-limited).
If Microsoft fails with HTTP 429, re-run this cell later; finished employers are skipped.

In [ ]:
from pipeline.employers import fetch_employers
fetch_employers()            # test first with: fetch_employers(only=['amazon'], terms=['forward deployed'])

Postings per company group and family (every group should have target-family postings):

In [ ]:
from pipeline.summary import coverage
coverage()

## 3. Wayback coverage test (go/no-go gate)

Archived posting URLs per period, and the share of sampled snapshots that are usable.
Check about 5 cached pages by eye before trusting the numbers.

In [ ]:
from pipeline.wayback import coverage as wayback_coverage
wb = wayback_coverage()
wb['estimate'].est_usable.unstack('period', fill_value=0)

In [ ]:
wb['by_year']

## 4. HN Who is Hiring

In [ ]:
from pipeline.hn import fetch_hn, monthly_check
hn = fetch_hn()
months, missing = monthly_check(hn)
print(f'{len(hn):,} comments; {len(months)} months present; missing: {missing[:24]}')
months.plot(figsize=(10, 3), title='HN Who is Hiring: comments per month');

## 5. Kaggle LinkedIn sets

In [ ]:
from pipeline.kaggle_data import fetch_kaggle
fetch_kaggle()

## 6. Summary for the go/no-go call

In [ ]:
from pipeline.summary import week1_summary
week1_summary()